<a href="https://colab.research.google.com/github/karye/Liu-labbar/blob/main/Gymnasiet_Lab_2_Maskininlarning/Lektion_6_Trana_Bedrageri_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Maskininlärning – Lektion 6: Träna Bedrägeridetektor-AI

**Målgrupp:** Gymnasiet, 16 år, inga förkunskaper krävs  
**Tid:** ca 45–50 minuter  
**Mål:** Träna en riktig AI-modell på bedrägeridata och förstå Förväxlingsmatrisen (Confusion Matrix)

> 📋 **Förutsättning:** Du har gjort Lektion 5 och förstår varför obalanserad data är ett problem.

---

### Upphovspersoner
Originalversion: David Bergström & Mattias Tiger, mattias.tiger@liu.se  
Gymnasieversion baserad på originalverket ovan.

### Licens
CC BY-NC-SA 4.0 – https://creativecommons.org/licenses/by-nc-sa/4.0/

---
## 🔁 Del 1 – Snabb repetition från Lektion 5

I förra lektionen lärde vi oss att:
- Av alla kreditkortstransaktioner är bara **0,17%** bedrägerier – resten är normala köp
- Det kallas **Obalanserad data (Imbalanced Data)**
- En modell som alltid svarar "Inte bedrägeri" kan ändå få **99,8% noggrannhet (Accuracy)** – men är helt värdelös

Nu ska vi ta nästa steg:
1. Ladda om bedrägeridatan
2. Dela upp den i träning och test  
3. Träna en riktig AI-modell
4. Använda en **Förväxlingsmatris (Confusion Matrix)** för att se vad AI:n faktiskt gör

*Det är som att ge AI:n ett prov och sedan gå igenom varje svar – inte bara räkna hur många den fick rätt!*

---
## 📥 Del 2 – Ladda in bedrägeridatan igen

Vi börjar med att installera och importera de verktyg vi behöver, och sedan laddar vi in datan.

> ⏳ Det kan ta upp till 60 sekunder att ladda datasetet – ha tålamod!

In [ ]:
!pip install xgboost -q
print("✅ XGBoost installerat!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
from xgboost import XGBClassifier

print("Laddar data från internet... (kan ta 30-60 sekunder)")

url = 'https://raw.githubusercontent.com/nsethi31/Kaggle-Data-Credit-Card-Fraud-Detection/refs/heads/master/creditcard.csv'
df = pd.read_csv(url)

print(f"\n✅ Data laddad!")
print(f"   Antal transaktioner: {len(df):,}")
print(f"   Antal bedrägerier:   {(df['Class']==1).sum():,} ({(df['Class']==1).mean():.2%})")

---
## ✂️ Del 3 – Dela upp datan: Träning och Test

Precis som när en lärare sätter ett prov ska AI:n inte ha sett testfrågorna under träningen.  
Det skulle vara som att ge eleverna facit innan provet – de "klarar" det, men vi vet inte om de faktiskt förstår!

Vi delar upp datan i två delar:
- **Träningsdata (Training Data):** 80% av datan – detta är AI:ns "studiematerial"
- **Testdata (Test Data):** 20% av datan – detta är AI:ns "prov" som den aldrig sett förut

```
100% av all data
    │
    ├── 80% Träning  ──→  AI:n lär sig mönster
    │
    └── 20% Test     ──→  Vi utvärderar hur bra AI:n är
```

In [ ]:
# Förbered funktioner (X) och svar (y)
X = df.drop(columns=['Class'])   # Allt utom "Class"-kolumnen är egenskaper
y = df['Class']                  # 'Class': 0 = normalt, 1 = bedrägeri

# Dela upp i träning (80%) och test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,     # 20% går till test
    random_state=42    # Slumpen fixeras så alla får samma uppdelning
)

print("✅ Datan delad!")
print(f"   Träningsdata:  {len(X_train):,} transaktioner ({len(X_train)/len(df):.0%})")
print(f"   Testdata:      {len(X_test):,} transaktioner ({len(X_test)/len(df):.0%})")
print()
print(f"   Bedrägerier i träningsdata: {y_train.sum():,}")
print(f"   Bedrägerier i testdata:     {y_test.sum():,}")

---
## 🌱 Del 4 – Träna AI-modellen (enkel version)

Nu tränar vi vår AI! Vi använder **XGBoost** – en kraftfull maskininlärningsmodell.

XGBoost bygger många små **beslutsträd (Decision Trees)** i rad.  
Varje träd lär sig av de misstag det föregående trädet gjorde, precis som när man övar inför ett prov:
- Träd 1: Försöker. Gör misstag.
- Träd 2: Fokuserar på misstagen. Förbättrar sig.
- Träd 3: Fokuserar på kvarvarande misstag. Förbättrar sig ytterligare.
- ... och så vidare!

Vi börjar med en **enkel** modell – lite som att bara öva snabbt innan provet:

In [ ]:
print("Tränar enkel modell (snabb men enkel)...")
print("(kan ta 30-60 sekunder)")

# Enkel modell: max_depth=1 betyder mycket grunt beslutsträd (ställer bara 1 fråga)
# n_estimators=1 betyder bara ett enda träd
modell_enkel = XGBClassifier(
    max_depth=1,         # Djupet på varje träd (1 = mycket enkelt)
    n_estimators=1,      # Antal träd
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)
modell_enkel.fit(X_train, y_train)

# Gör förutsägelser på testdatan
y_pred_enkel = modell_enkel.predict(X_test)

noggrannhet_enkel = accuracy_score(y_test, y_pred_enkel)
print(f"\n✅ Enkel modellens noggrannhet (Accuracy): {noggrannhet_enkel:.2%}")
print()
print("📊 Imponerande siffra – men låt oss titta djupare...")

---
## 📊 Del 5 – Förväxlingsmatrisen (Confusion Matrix)

Noggrannhet säger inte hela sanningen. Vi behöver **Förväxlingsmatrisen (Confusion Matrix)** –  
ett rutnät som visar exakt vilka fel AI:n gör.

### De fyra rutorna i matrisen

```
                        AI:ns gissning:
                    NORMALT        BEDRÄGERI
              ┌──────────────┬──────────────┐
Verkligt svar │  Sant Negativ│  Falskt      │
NORMALT        │  (True Neg.) │  Positiv     │
              │  ✅ Rätt!    │  ⚠️  Falskt  │
              │              │  larm         │
              ├──────────────┼──────────────┤
Verkligt svar │  Falskt      │  Sant Positiv│
BEDRÄGERI      │  Negativ     │  (True Pos.) │
              │  ❌ Missad   │  ✅ Rätt!    │
              │  tjuv!        │              │
              └──────────────┴──────────────┘
```

- 🟢 **Sant Negativ (True Negative, TN):** Normalt köp – AI sa Normalt → **Rätt!**
- 🟢 **Sant Positiv (True Positive, TP):** Bedrägeri – AI sa Bedrägeri → **Rätt!**
- 🟡 **Falskt Positiv (False Positive, FP):** Normalt köp – AI sa Bedrägeri → **Falskt larm** *(irriterande för kunden)*
- 🔴 **Falskt Negativ (False Negative, FN):** Bedrägeri – AI sa Normalt → **Tjuven slipper undan!** *(farligt!)*

> 🎯 Det allvarligaste felet är **Falskt Negativ** – när AI:n missar ett riktigt bedrägeri!

In [ ]:
# Rita förväxlingsmatris för den enkla modellen
fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_enkel,
    display_labels=['Normalt', 'Bedrägeri'],
    colorbar=False,
    ax=ax
)
ax.set_title('Enkel AI-modell\n(max_depth=1, 1 träd)', fontsize=12)
plt.tight_layout()
plt.show()

# Räkna specifikt
tp = ((y_pred_enkel == 1) & (y_test == 1)).sum()
fn = ((y_pred_enkel == 0) & (y_test == 1)).sum()
fp = ((y_pred_enkel == 1) & (y_test == 0)).sum()
tn = ((y_pred_enkel == 0) & (y_test == 0)).sum()
totalt_bedragerier = y_test.sum()

print(f"Totalt bedrägerier i testdata: {totalt_bedragerier}")
print(f"✅ Sant Positiv (hittade bedrägerier):    {tp} ({tp/totalt_bedragerier:.1%})")
print(f"❌ Falskt Negativ (missade bedrägerier):  {fn} ({fn/totalt_bedragerier:.1%})")
print(f"⚠️  Falskt Positiv (falska larm):          {fp}")

### 💬 Reflektionsfråga 6.1

Titta på förväxlingsmatrisen för den enkla modellen.

**Fråga 1:** Hur många bedrägerier hittade AI:n? Hur många missade den?

**Fråga 2:** Om du var offer för ett bedrägeri och din bank använde den här AI:n –  
vad är sannolikheten att bedrägeriet stoppas?

**Fråga 3:** Är det acceptabelt? Vad tycker du?

---
## 🚀 Del 6 – En bättre modell: Mer träning, djupare träd

Den enkla modellen hade problem. Låt oss träna en **bättre** modell!  
Vi ökar antalet träd och gör varje träd djupare (fler ja/nej-frågor per träd):

- Förra modellen: 1 träd, 1 fråga per träd → Mycket enkel
- Ny modell: 10 träd, 6 frågor per träd → Mycket mer sofistikerad

In [ ]:
print("Tränar bättre modell (mer komplex, mer tid)...")
print("(kan ta 60-120 sekunder)")

# Bättre modell: fler träd och djupare beslutsträd
modell_battre = XGBClassifier(
    max_depth=6,         # Djupare träd = fler frågor = mer komplex
    n_estimators=10,     # Fler träd = AI:n gör fler försök att lära sig
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)
modell_battre.fit(X_train, y_train)

# Gör förutsägelser
y_pred_battre = modell_battre.predict(X_test)

noggrannhet_battre = accuracy_score(y_test, y_pred_battre)
print(f"\n✅ Bättre modellens noggrannhet (Accuracy): {noggrannhet_battre:.2%}")

---
## 🔍 Del 7 – Jämför de två modellerna

Nu jämför vi den enkla och den bättre modellen sida vid sida.  
Kom ihåg: hög noggrannhet ≠ bra på att hitta bedrägerier!

In [ ]:
# Jämför förväxlingsmatriser för båda modellerna
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Enkel modell
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_enkel,
    display_labels=['Normalt', 'Bedrägeri'],
    colorbar=False,
    ax=axes[0]
)
axes[0].set_title(f'Enkel modell\n(Accuracy: {accuracy_score(y_test, y_pred_enkel):.2%})', fontsize=11)

# Bättre modell
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_battre,
    display_labels=['Normalt', 'Bedrägeri'],
    colorbar=False,
    ax=axes[1]
)
axes[1].set_title(f'Bättre modell\n(Accuracy: {accuracy_score(y_test, y_pred_battre):.2%})', fontsize=11)

plt.suptitle('Jämförelse – Förväxlingsmatriser (Confusion Matrices)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Djupare analys av den bättre modellen
totalt_bedragerier = y_test.sum()
tp = ((y_pred_battre == 1) & (y_test == 1)).sum()
fn = ((y_pred_battre == 0) & (y_test == 1)).sum()
fp = ((y_pred_battre == 1) & (y_test == 0)).sum()
tn = ((y_pred_battre == 0) & (y_test == 0)).sum()

print("=== Resultat: Bättre modell ===")
print(f"Totalt bedrägerier i testdata:    {totalt_bedragerier}")
print()
print(f"✅ Sant Positiv (TP) – hittade:   {tp:>4}  ({tp/totalt_bedragerier:.1%} av alla bedrägerier)")
print(f"❌ Falskt Negativ (FN) – missade: {fn:>4}  ({fn/totalt_bedragerier:.1%} av alla bedrägerier)")
print(f"⚠️  Falskt Positiv (FP) – falska larm: {fp:>4} (normala köp felaktigt flaggade)")
print(f"🟢 Sant Negativ (TN) – rätt normala: {tn:>6}")
print()
print("=== Jämförelse ===")
tp_enkel = ((y_pred_enkel == 1) & (y_test == 1)).sum()
fn_enkel = ((y_pred_enkel == 0) & (y_test == 1)).sum()
print(f"Enkel modell hittade:  {tp_enkel} bedrägerier ({tp_enkel/totalt_bedragerier:.1%})")
print(f"Bättre modell hittade: {tp} bedrägerier ({tp/totalt_bedragerier:.1%})")

---
## 💬 Del 8 – Diskussion: Är modellen bra nog?

Baserat på förväxlingsmatrisen för testdata:

### Fråga 1: Vad innebär detta för kunder som utsätts för bedrägeri?
- Om du är ett offer för bedrägeri – vad är chansen att AI:n stoppar det?
- Är det acceptabelt att en så stor andel bedrägerier fortfarande missar?

### Fråga 2: Vad innebär detta för vanliga kunder?
- Vad är sannolikheten att ett normalt köp felaktigt stoppas (Falskt Positiv)?
- Hur tror du en kund reagerar när deras kort spärras mitt i ett köp?

### Fråga 3: Vad tycker du om resultatet?
- Skulle du vilja att din nuvarande bank använde denna modell?
- Finns det situationer där du *hellre* vill att AI:n är mer eller mindre aggressiv mot bedrägerier?

> 💡 **Nyckelinsikt:** Den bättre modellen är tydligt bättre än den enkla – men vi missar fortfarande  
> många bedrägerier. Anledningen? **Obalansen i datan!** AI:n har sett enormt många normala köp  
> men väldigt få bedrägerier under träningen, så den är "van vid" normala köp.  
>  
> I **Lektion 7** lär vi oss tekniker för att fixa detta problem!

---
## 🎓 Bra jobbat – du är klar med Lektion 6!

### Vad du lärt dig idag:

| Begrepp | Förklaring | Engelskt namn |
|---------|------------|---------------|
| **Träningsdata** | Data AI:n lär sig av (80%) | Training Data |
| **Testdata** | Data AI:n utvärderas på (20%) | Test Data |
| **XGBoost** | En kraftfull AI-modell av beslutsträd | XGBoost |
| **Förväxlingsmatris** | Rutnät som visar vilka fel AI:n gör | Confusion Matrix |
| **Sant Positiv (TP)** | Bedrägeri hittat – rätt! | True Positive |
| **Sant Negativ (TN)** | Normalt köp korrekt godkänt | True Negative |
| **Falskt Positiv (FP)** | Normalt köp felaktigt stoppat | False Positive |
| **Falskt Negativ (FN)** | Bedrägeri som slipper igenom – farligt! | False Negative |

### 👉 I Lektion 7 ska vi:
1. Lära oss hur man balanserar obalanserad data
2. Träna en förbättrad modell som faktiskt hittar fler bedrägerier
3. Introducera bättre mätvärden: **Precision** och **Recall (Känslighet)**
4. Diskutera avvägningarna – kan man ha *för* bra bedrägeridetektor?